In [1]:
import os

print(os.path.exists("/content/beijing_air_quality_cleaned.csv"))

True


In [2]:
!pip install -q streamlit plotly

   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 10.1/10.1 MB 22.9 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 11.4/11.4 MB 52.6 MB/s eta 0:00:00


In [16]:
%%writefile /content/app.py

import streamlit as st
import pandas as pd
import plotly.express as px


# ============================================================
# PAGE CONFIGURATION
# ============================================================

st.set_page_config(
    page_title="Beijing Air Quality Dashboard",
    page_icon="🌍",
    layout="wide"
)


# ============================================================
# LOAD DATA
# ============================================================

@st.cache_data
def load_data():

    df = pd.read_csv("/content/beijing_air_quality_cleaned.csv")

    df["datetime"] = pd.to_datetime(df["datetime"])

    return df


df = load_data()


# ============================================================
# TITLE
# ============================================================

st.title("🌍 Beijing Air Quality Intelligence Dashboard")

st.write(
    "Interactive analysis of hourly air-quality data from "
    "12 monitoring stations in Beijing."
)


# ============================================================
# SIDEBAR FILTERS
# ============================================================

st.sidebar.header("🎛️ Dashboard Filters")

stations = sorted(df["station"].unique())

selected_station = st.sidebar.selectbox(
    "Select Station",
    ["All Stations"] + stations
)

pollutants = [
    "PM2.5",
    "PM10",
    "SO2",
    "NO2",
    "CO",
    "O3"
]

selected_pollutant = st.sidebar.selectbox(
    "Select Pollutant",
    pollutants
)

years = sorted(df["year"].unique())

selected_year = st.sidebar.selectbox(
    "Select Year",
    ["All Years"] + years
)


# ============================================================
# FILTER DATA
# ============================================================

filtered_df = df.copy()


if selected_station != "All Stations":

    filtered_df = filtered_df[
        filtered_df["station"] == selected_station
    ]


if selected_year != "All Years":

    filtered_df = filtered_df[
        filtered_df["year"] == selected_year
    ]


# ============================================================
# KEY METRICS
# ============================================================

st.subheader("📊 Air Quality Summary")


average_value = filtered_df[selected_pollutant].mean()

minimum_value = filtered_df[selected_pollutant].min()

maximum_value = filtered_df[selected_pollutant].max()

record_count = len(filtered_df)


col1, col2, col3, col4 = st.columns(4)


col1.metric(
    "Average",
    f"{average_value:.2f}"
)

col2.metric(
    "Minimum",
    f"{minimum_value:.2f}"
)

col3.metric(
    "Maximum",
    f"{maximum_value:.2f}"
)

col4.metric(
    "Records",
    f"{record_count:,}"
)


# ============================================================
# OVERALL AIR QUALITY TREND
# ============================================================

st.subheader("🌫️ Overall Air Quality Trend")


# ------------------------------------------------------------
# CPCB AQI BREAKPOINTS
# ------------------------------------------------------------

aqi_breakpoints = {

    "PM2.5": [
        (0, 30, 0, 50),
        (31, 60, 51, 100),
        (61, 90, 101, 200),
        (91, 120, 201, 300),
        (121, 250, 301, 400),
        (251, 380, 401, 500)
    ],

    "PM10": [
        (0, 50, 0, 50),
        (51, 100, 51, 100),
        (101, 250, 101, 200),
        (251, 350, 201, 300),
        (351, 430, 301, 400),
        (431, 510, 401, 500)
    ],

    "NO2": [
        (0, 40, 0, 50),
        (41, 80, 51, 100),
        (81, 180, 101, 200),
        (181, 280, 201, 300),
        (281, 400, 301, 400),
        (401, 520, 401, 500)
    ],

    "SO2": [
        (0, 40, 0, 50),
        (41, 80, 51, 100),
        (81, 380, 101, 200),
        (381, 800, 201, 300),
        (801, 1600, 301, 400),
        (1601, 2620, 401, 500)
    ],

    "CO": [
        (0, 1, 0, 50),
        (1.1, 2, 51, 100),
        (2.1, 10, 101, 200),
        (10.1, 17, 201, 300),
        (17.1, 34, 301, 400),
        (34.1, 50, 401, 500)
    ],

    "O3": [
        (0, 50, 0, 50),
        (51, 100, 51, 100),
        (101, 168, 101, 200),
        (169, 208, 201, 300),
        (209, 748, 301, 400),
        (749, 1000, 401, 500)
    ]
}


# ------------------------------------------------------------
# AQI SUB-INDEX FUNCTION
# ------------------------------------------------------------

def calculate_subindex(concentration, pollutant):

    if pd.isna(concentration):
        return None

    for c_low, c_high, i_low, i_high in aqi_breakpoints[pollutant]:

        if c_low <= concentration <= c_high:

            index = (
                (i_high - i_low)
                / (c_high - c_low)
                * (concentration - c_low)
                + i_low
            )

            return round(index, 2)

    return None


# ------------------------------------------------------------
# AQI CALCULATION FOR A DATAFRAME
# ------------------------------------------------------------

def calculate_overall_aqi(data):

    pollutant_subindices = {}

    for pollutant in aqi_pollutants:

        concentration = data[pollutant].mean()

        subindex = calculate_subindex(
            concentration,
            pollutant
        )

        if subindex is not None:

            pollutant_subindices[pollutant] = subindex

    if not pollutant_subindices:

        return None, None

    overall_aqi = max(
        pollutant_subindices.values()
    )

    dominant_pollutant = max(
        pollutant_subindices,
        key=pollutant_subindices.get
    )

    return overall_aqi, dominant_pollutant


# ------------------------------------------------------------
# POLLUTANTS USED FOR AQI
# ------------------------------------------------------------

aqi_pollutants = [
    "PM2.5",
    "PM10",
    "NO2",
    "SO2",
    "CO",
    "O3"
]


# ------------------------------------------------------------
# CALCULATE AQI OVER TIME
# ------------------------------------------------------------

if selected_year == "All Years":

    # Yearly AQI
    aqi_data = []

    for year, group in filtered_df.groupby("year"):

        aqi, dominant = calculate_overall_aqi(group)

        if aqi is not None:

            aqi_data.append({
                "Period": str(year),
                "AQI": aqi,
                "Dominant Pollutant": dominant
            })

    aqi_df = pd.DataFrame(aqi_data)

    chart_title = "Yearly Average AQI"

else:

    # Monthly AQI
    aqi_data = []

    filtered_df["month_period"] = (
        filtered_df["datetime"]
        .dt.to_period("M")
    )

    for period, group in filtered_df.groupby("month_period"):

        aqi, dominant = calculate_overall_aqi(group)

        if aqi is not None:

            aqi_data.append({
                "Period": period.strftime("%b %Y"),
                "AQI": aqi,
                "Dominant Pollutant": dominant
            })

    aqi_df = pd.DataFrame(aqi_data)

    chart_title = f"Monthly Average AQI — {selected_year}"


# ------------------------------------------------------------
# DISPLAY AQI TREND
# ------------------------------------------------------------

if not aqi_df.empty:

    fig_aqi = px.line(
        aqi_df,
        x="Period",
        y="AQI",
        markers=True,
        title=chart_title
    )

    fig_aqi.update_layout(
        xaxis_title="Year" if selected_year == "All Years" else "Month",
        yaxis_title="AQI"
    )

    st.plotly_chart(
        fig_aqi,
        use_container_width=True
    )


# ------------------------------------------------------------
# AQI SUMMARY
# ------------------------------------------------------------

if not aqi_df.empty:

    average_aqi = aqi_df["AQI"].mean()

    highest_aqi = aqi_df["AQI"].max()

    highest_aqi_period = aqi_df.loc[
        aqi_df["AQI"].idxmax(),
        "Period"
    ]

    dominant_pollutant = (
        aqi_df["Dominant Pollutant"]
        .mode()[0]
    )


    col1, col2, col3 = st.columns(3)


    col1.metric(
        "Average AQI",
        f"{average_aqi:.0f}"
    )

    col2.metric(
        "Highest AQI",
        f"{highest_aqi:.0f}"
    )

    col3.metric(
        "Dominant Pollutant",
        dominant_pollutant
    )


    st.info(
        f"The highest AQI occurred during "
        f"**{highest_aqi_period}**, with an AQI of "
        f"**{highest_aqi:.0f}**."
    )


st.caption(
    "AQI-style estimate using CPCB breakpoint ranges. "
    "The dataset contains hourly observations, so this is "
    "an analytical estimate rather than an official CPCB AQI."
)

# ============================================================
# DAILY TREND
# ============================================================

st.subheader(
    f"📈 Daily {selected_pollutant} Trend"
)


daily_data = (

    filtered_df
    .set_index("datetime")[selected_pollutant]
    .resample("D")
    .mean()
    .reset_index()

)


fig_daily = px.line(

    daily_data,

    x="datetime",

    y=selected_pollutant,

    title=f"Daily Average {selected_pollutant}"

)


fig_daily.update_layout(

    xaxis_title="Date",

    yaxis_title=selected_pollutant

)


st.plotly_chart(

    fig_daily,

    use_container_width=True

)


# ============================================================
# MONTHLY TREND
# ============================================================

st.subheader(
    f"📅 Monthly {selected_pollutant} Trend"
)


monthly_data = (

    filtered_df
    .set_index("datetime")[selected_pollutant]
    .resample("ME")
    .mean()
    .reset_index()

)


fig_monthly = px.line(

    monthly_data,

    x="datetime",

    y=selected_pollutant,

    markers=True,

    title=f"Monthly Average {selected_pollutant}"

)


fig_monthly.update_layout(

    xaxis_title="Month",

    yaxis_title=selected_pollutant

)


st.plotly_chart(

    fig_monthly,

    use_container_width=True

)


# ============================================================
# HOURLY PATTERN + DAY OF WEEK
# ============================================================

col1, col2 = st.columns(2)


# ---------------- HOURLY ----------------

with col1:

    st.subheader("⏰ Hourly Pattern")


    hourly_data = (

        filtered_df
        .groupby("hour")[selected_pollutant]
        .mean()
        .reset_index()

    )


    fig_hour = px.line(

        hourly_data,

        x="hour",

        y=selected_pollutant,

        markers=True,

        title=f"Average {selected_pollutant} by Hour"

    )


    fig_hour.update_layout(

        xaxis_title="Hour of Day",

        yaxis_title=selected_pollutant

    )


    st.plotly_chart(

        fig_hour,

        use_container_width=True

    )


# ---------------- DAY OF WEEK ----------------

with col2:

    st.subheader("📅 Day-of-Week Pattern")


    day_order = [

        "Monday",
        "Tuesday",
        "Wednesday",
        "Thursday",
        "Friday",
        "Saturday",
        "Sunday"

    ]


    weekday_data = (

        filtered_df
        .groupby("day_of_week")[selected_pollutant]
        .mean()
        .reindex(day_order)
        .reset_index()

    )


    fig_weekday = px.bar(

        weekday_data,

        x="day_of_week",

        y=selected_pollutant,

        title=f"Average {selected_pollutant} by Day"

    )


    fig_weekday.update_layout(

        xaxis_title="Day",

        yaxis_title=selected_pollutant

    )


    st.plotly_chart(

        fig_weekday,

        use_container_width=True

    )


# ============================================================
# POLLUTANT COMPARISON
# ============================================================

st.subheader("🧪 Pollutant Comparison")


pollutant_avg = (

    filtered_df[pollutants]
    .mean()
    .reset_index()

)


pollutant_avg.columns = [

    "Pollutant",
    "Average"

]


fig_pollutants = px.bar(

    pollutant_avg,

    x="Pollutant",

    y="Average",

    title="Average Pollutant Levels"

)


fig_pollutants.update_layout(

    xaxis_title="Pollutant",

    yaxis_title="Average Level"

)


st.plotly_chart(

    fig_pollutants,

    use_container_width=True

)


# ============================================================
# STATION COMPARISON
# ============================================================

st.subheader(
    f"📍 Average {selected_pollutant} by Station"
)


station_data = (

    df
    .groupby("station")[selected_pollutant]
    .mean()
    .reset_index()
    .sort_values(
        selected_pollutant,
        ascending=False
    )

)


fig_station = px.bar(

    station_data,

    x="station",

    y=selected_pollutant,

    title=f"Average {selected_pollutant} Across Stations"

)


fig_station.update_layout(

    xaxis_title="Station",

    yaxis_title=selected_pollutant

)


st.plotly_chart(

    fig_station,

    use_container_width=True

)

# ============================================================
# WEATHER VS POLLUTION
# ============================================================

st.subheader("🌦️ Weather vs Pollution")

weather_col1, weather_col2 = st.columns(2)


# ---------------- WIND SPEED ----------------

with weather_col1:

    st.write(f"**Wind Speed vs {selected_pollutant}**")

    wind_sample = filtered_df.sample(
        min(5000, len(filtered_df)),
        random_state=42
    )

    fig_wind = px.scatter(
        wind_sample,
        x="WSPM",
        y=selected_pollutant,
        opacity=0.5,
        title=f"Wind Speed vs {selected_pollutant}"
    )

    fig_wind.update_layout(
        xaxis_title="Wind Speed (WSPM)",
        yaxis_title=selected_pollutant
    )

    st.plotly_chart(
        fig_wind,
        use_container_width=True
    )


# ---------------- TEMPERATURE ----------------

with weather_col2:

    st.write(f"**Temperature vs {selected_pollutant}**")

    temp_sample = filtered_df.sample(
        min(5000, len(filtered_df)),
        random_state=42
    )

    fig_temp = px.scatter(
        temp_sample,
        x="TEMP",
        y=selected_pollutant,
        opacity=0.5,
        title=f"Temperature vs {selected_pollutant}"
    )

    fig_temp.update_layout(
        xaxis_title="Temperature",
        yaxis_title=selected_pollutant
    )

    st.plotly_chart(
        fig_temp,
        use_container_width=True
    )

# ============================================================
# CORRELATION MATRIX
# ============================================================

st.subheader("🔗 Pollutant Correlation")


correlation = filtered_df[pollutants].corr()


fig_corr = px.imshow(

    correlation,

    text_auto=True,

    aspect="auto",

    title="Pollutant Correlation Matrix"

)


st.plotly_chart(

    fig_corr,

    use_container_width=True

)


# ============================================================
# BEST AND WORST DAYS
# ============================================================

st.subheader("🏆 Best and Worst Days")


daily_pollution = (

    filtered_df
    .set_index("datetime")[selected_pollutant]
    .resample("D")
    .mean()
    .dropna()

)


if len(daily_pollution) > 0:

    best_date = daily_pollution.idxmin()

    best_value = daily_pollution.min()

    worst_date = daily_pollution.idxmax()

    worst_value = daily_pollution.max()


    col1, col2 = st.columns(2)


    with col1:

        st.metric(

            "Lowest Pollution Day",

            best_date.strftime("%d %b %Y"),

            f"{best_value:.2f}"

        )


    with col2:

        st.metric(

            "Highest Pollution Day",

            worst_date.strftime("%d %b %Y"),

            f"{worst_value:.2f}"

        )


# ============================================================
# AUTOMATIC INSIGHTS
# ============================================================

st.subheader("💡 Key Insights")


# Monthly insight

monthly_avg = (

    filtered_df
    .groupby("month")[selected_pollutant]
    .mean()

)


highest_month = monthly_avg.idxmax()

highest_month_value = monthly_avg.max()


# Hourly insight

hourly_avg = (

    filtered_df
    .groupby("hour")[selected_pollutant]
    .mean()

)


highest_hour = hourly_avg.idxmax()

highest_hour_value = hourly_avg.max()


# Station insight

station_avg = (

    df
    .groupby("station")[selected_pollutant]
    .mean()

)


highest_station = station_avg.idxmax()

highest_station_value = station_avg.max()


st.write(

    f"• The average **{selected_pollutant}** level is "
    f"**{average_value:.2f}**."

)


st.write(

    f"• The highest average monthly level occurred in "
    f"**month {highest_month}**, with a value of "
    f"**{highest_month_value:.2f}**."

)


st.write(

    f"• The highest average hourly level occurred around "
    f"**{highest_hour}:00**, with a value of "
    f"**{highest_hour_value:.2f}**."

)


st.write(

    f"• Across all stations, **{highest_station}** has the "
    f"highest average {selected_pollutant}, at "
    f"**{highest_station_value:.2f}**."

)


# ============================================================
# DATA PREVIEW
# ============================================================

with st.expander("🔍 View Cleaned Data"):

    st.dataframe(

        filtered_df.head(100),

        use_container_width=True

    )


# ============================================================
# PROJECT ROADMAP
# ============================================================

st.sidebar.markdown("---")

st.sidebar.subheader("🚀 Project Roadmap")

st.sidebar.write(

    """
    ✅ Data Collection

    ✅ Data Cleaning

    ✅ Exploratory Data Analysis

    ✅ Data Visualization

    ✅ Streamlit Dashboard

    ⏳ SQL Analysis

    ⏳ Machine Learning

    ⏳ Deep Learning

    ⏳ Agentic AI
    """

)


# ============================================================
# FOOTER
# ============================================================

st.markdown("---")

st.caption(

    "Beijing Multi-Site Air Quality Analysis | "
    "Python • Pandas • Plotly • Streamlit"

)

Overwriting /content/app.py


In [10]:
!streamlit run /content/app.py --server.port 8501 --server.address 0.0.0.0 > /content/streamlit.log 2>&1 &

In [11]:
!wget -q https://github.com/cloudflare/cloudflared/releases/latest/download/cloudflared-linux-amd64 -O /content/cloudflared
!chmod +x /content/cloudflared

/content/cloudflared: Text file busy


In [12]:
!nohup /content/cloudflared tunnel --url http://localhost:8501 > /content/cloudflared.log 2>&1 &

In [13]:
import time
import re

time.sleep(5)

with open("/content/cloudflared.log") as f:
    log = f.read()

urls = re.findall(
    r"https://[a-zA-Z0-9.-]+\.trycloudflare\.com",
    log
)

if urls:
    print("🌍 DASHBOARD:")
    print(urls[-1])
else:
    print(log)

🌍 DASHBOARD:
https://switching-nhs-indicated-receptor.trycloudflare.com
